# I2S VAD Research — Model Evaluation

Evaluates all four models for a given experiment.

**To switch experiment:** change `EXPERIMENT` in Cell 1.

Available experiments:
- `exp1_alaw_synthetic` — A-law + synthetic noise (LG Gram test)
- `exp2_clean_musan`    — Clean PCM + real MUSAN (MSI baseline)
- `exp3_alaw_musan`    — A-law + real MUSAN (MSI main contribution)

**Run all cells top to bottom.**

In [ ]:
# ── Configuration — change EXPERIMENT here ────────────────────────────────
EXPERIMENT = "exp1_alaw_synthetic"
# EXPERIMENT = "exp2_clean_musan"
# EXPERIMENT = "exp3_alaw_musan"

import sys
import time
import json
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 150
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix
)

# paths
ROOT = Path('').resolve()
if 'notebooks' in str(ROOT):
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.models.cnn1d           import CNN1D
from src.models.wavenet_small   import WaveNetSmall
from src.models.ecapa_vad       import ECAPAVAD
from src.models.transformer_vad import TransformerVAD

SPLITS     = ROOT / 'data'    / 'splits'  / EXPERIMENT
MODELS_DIR = ROOT / 'outputs' / EXPERIMENT / 'models'
STATS_DIR  = ROOT / 'outputs' / EXPERIMENT / 'stats'
FIGS_DIR   = ROOT / 'outputs' / EXPERIMENT / 'figures'
FIGS_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Experiment : {EXPERIMENT}')
print(f'Device     : {device}')
print(f'Models dir : {MODELS_DIR}')

In [ ]:
# ── Load test set ──────────────────────────────────────────────────────────
X_test = np.load(str(SPLITS / 'X_test.npy'))
y_test = np.load(str(SPLITS / 'y_test.npy'))
X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)

print(f'Test set : X {X_test.shape}  y {y_test.shape}')
print(f'Speech   : {y_test.sum()}  |  Noise : {(y_test==0).sum()}')

In [ ]:
# ── Load models ───────────────────────────────────────────────────────────
MODEL_REGISTRY = {
    'CNN1D':          (CNN1D(num_classes=2),          'CNN1D_best.pt'),
    'WaveNetSmall':   (WaveNetSmall(num_classes=2),   'WaveNetSmall_best.pt'),
    'ECAPAVAD':       (ECAPAVAD(num_classes=2),       'ECAPAVAD_best.pt'),
    'TransformerVAD': (TransformerVAD(num_classes=2), 'TransformerVAD_best.pt'),
}

def load_model(model, filename):
    path = MODELS_DIR / filename
    ckpt = torch.load(str(path), map_location=device)
    model.load_state_dict(ckpt['model_state'])
    model = model.to(device)
    model.eval()
    print(f'  {filename:35s}  epoch={ckpt["epoch"]:3d}  val_acc={ckpt["val_acc"]:.4f}')
    return model

print('Loading models...')
loaded_models = {}
for name, (model, fname) in MODEL_REGISTRY.items():
    try:
        loaded_models[name] = load_model(model, fname)
    except FileNotFoundError:
        print(f'  [skip] {fname} not found')
print(f'Loaded {len(loaded_models)} models.')

In [ ]:
# ── Evaluate all models ───────────────────────────────────────────────────
LABELS    = ['Noise', 'Speech']
COLORS    = ['steelblue', 'seagreen', 'tomato', 'mediumpurple']
N_LATENCY = 200
results   = []

for name, model in loaded_models.items():
    print(f'\nEvaluating {name}...')

    with torch.no_grad():
        preds = model(X_test_t).argmax(1).cpu().numpy()

    acc  = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, average='macro', zero_division=0)
    rec  = recall_score(y_test, preds, average='macro', zero_division=0)
    f1   = f1_score(y_test, preds, average='macro', zero_division=0)
    f1pc = f1_score(y_test, preds, average=None, zero_division=0)

    # latency
    single = X_test_t[0:1]
    for _ in range(10):
        with torch.no_grad(): _ = model(single)
    t0 = time.perf_counter()
    for _ in range(N_LATENCY):
        with torch.no_grad(): _ = model(single)
    latency_ms = (time.perf_counter() - t0) / N_LATENCY * 1000

    pt_path  = MODELS_DIR / MODEL_REGISTRY[name][1]
    size_kb  = pt_path.stat().st_size / 1024
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f'  Accuracy  : {acc:.4f}')
    print(f'  F1 macro  : {f1:.4f}')
    print(f'  Latency   : {latency_ms:.3f} ms')
    print(f'  Size      : {size_kb:.1f} KB')

    results.append({
        'experiment':  EXPERIMENT,
        'model':       name,
        'accuracy':    round(acc,  4),
        'precision':   round(prec, 4),
        'recall':      round(rec,  4),
        'f1_macro':    round(f1,   4),
        'f1_noise':    round(float(f1pc[0]), 4),
        'f1_speech':   round(float(f1pc[1]), 4),
        'latency_ms':  round(latency_ms, 3),
        'size_kb':     round(size_kb, 1),
        'n_params':    n_params,
    })

df = pd.DataFrame(results)
csv_path = STATS_DIR / 'final_metrics.csv'
df.to_csv(str(csv_path), index=False)
print(f'\nSaved: {csv_path}')
print(df[['model','accuracy','f1_macro','latency_ms','size_kb','n_params']].to_string(index=False))

In [ ]:
# ── Figure 1: Confusion Matrices ──────────────────────────────────────────
n     = len(loaded_models)
fig, axes = plt.subplots(1, n, figsize=(5*n, 4))
if n == 1: axes = [axes]
fig.suptitle(f'Confusion Matrices — {EXPERIMENT}', fontsize=13, fontweight='bold')

for ax, (name, model) in zip(axes, loaded_models.items()):
    with torch.no_grad():
        preds = model(X_test_t).argmax(1).cpu().numpy()
    cm  = confusion_matrix(y_test, preds)
    row = df[df['model'] == name].iloc[0]
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=LABELS, yticklabels=LABELS,
                ax=ax, cbar=False)
    ax.set_title(f'{name}\nAcc={row["accuracy"]:.4f}  F1={row["f1_macro"]:.4f}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

plt.tight_layout()
path = FIGS_DIR / 'confusion_matrices.pdf'
plt.savefig(str(path), bbox_inches='tight')
print(f'Saved: {path}')
plt.show()

In [ ]:
# ── Figure 2: Training curves ─────────────────────────────────────────────
name_to_hist = {
    'CNN1D':          'CNN1D_history.json',
    'WaveNetSmall':   'WaveNetSmall_history.json',
    'ECAPAVAD':       'ECAPAVAD_history.json',
    'TransformerVAD': 'TransformerVAD_history.json',
}

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle(f'Training Curves — {EXPERIMENT}', fontsize=13, fontweight='bold')
axes = axes.flatten()

for ax, (name, fname) in zip(axes, name_to_hist.items()):
    hist_path = STATS_DIR / fname
    if not hist_path.exists():
        ax.set_title(f'{name} (no history)')
        continue
    with open(str(hist_path)) as f:
        h = json.load(f)
    ep = range(1, len(h['train_loss']) + 1)
    ax.plot(ep, h['train_loss'], label='Train loss', color='steelblue')
    ax.plot(ep, h['val_loss'],   label='Val loss',   color='tomato')
    ax2 = ax.twinx()
    ax2.plot(ep, h['val_acc'], label='Val acc', color='seagreen',
             linestyle='--', alpha=0.7)
    ax2.set_ylim(0, 1.05)
    ax2.set_ylabel('Val Accuracy', color='seagreen')
    ax.set_title(name)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
path = FIGS_DIR / 'training_curves.pdf'
plt.savefig(str(path), bbox_inches='tight')
print(f'Saved: {path}')
plt.show()

In [ ]:
# ── Figure 3: Model comparison bar chart ─────────────────────────────────
metrics    = ['accuracy', 'precision', 'recall', 'f1_macro']
bar_labels = ['Accuracy', 'Precision', 'Recall', 'F1 Macro']
x          = np.arange(len(metrics))
width      = 0.2

fig, ax = plt.subplots(figsize=(11, 5))
for i, (_, row) in enumerate(df.iterrows()):
    vals = [row[m] for m in metrics]
    bars = ax.bar(x + i*width, vals, width,
                  label=row['model'], color=COLORS[i % len(COLORS)], alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(bar_labels)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title(f'Model Comparison — {EXPERIMENT}', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
path = FIGS_DIR / 'model_comparison_bar.pdf'
plt.savefig(str(path), bbox_inches='tight')
print(f'Saved: {path}')
plt.show()

In [ ]:
# ── Figure 4: Accuracy vs Latency vs Size ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'Edge Deployment Tradeoff — {EXPERIMENT}',
             fontsize=13, fontweight='bold')

for ax, (xcol, xlabel) in zip(axes, [
    ('latency_ms', 'Inference Latency (ms)'),
    ('size_kb',    'Model Size (KB)'),
]):
    for i, (_, row) in enumerate(df.iterrows()):
        ax.scatter(row[xcol], row['accuracy'],
                   s=250, color=COLORS[i % len(COLORS)],
                   zorder=5, label=row['model'])
        ax.annotate(row['model'],
                    (row[xcol], row['accuracy']),
                    textcoords='offset points',
                    xytext=(8, 4), fontsize=8)
    if xcol == 'latency_ms':
        ax.axvline(x=32, color='red', linestyle='--',
                   alpha=0.5, label='32ms real-time limit')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Test Accuracy')
    ax.set_title(f'Accuracy vs {xlabel}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
path = FIGS_DIR / 'tradeoff_analysis.pdf'
plt.savefig(str(path), bbox_inches='tight')
print(f'Saved: {path}')
plt.show()

In [ ]:
# ── Figure 5: Summary table ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 2.5 + len(df) * 0.5))
ax.axis('off')

cols = ['Model', 'Accuracy', 'Precision', 'Recall', 'F1 Macro',
        'F1 Noise', 'F1 Speech', 'Latency (ms)', 'Size (KB)', 'Params']

data = []
for _, row in df.iterrows():
    data.append([
        row['model'],
        f"{row['accuracy']:.4f}",
        f"{row['precision']:.4f}",
        f"{row['recall']:.4f}",
        f"{row['f1_macro']:.4f}",
        f"{row['f1_noise']:.4f}",
        f"{row['f1_speech']:.4f}",
        f"{row['latency_ms']:.3f}",
        f"{row['size_kb']:.1f}",
        f"{row['n_params']:,}",
    ])

tbl = ax.table(cellText=data, colLabels=cols,
               cellLoc='center', loc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1.2, 2.0)

for j in range(len(cols)):
    tbl[0, j].set_facecolor('#2c3e50')
    tbl[0, j].set_text_props(color='white', fontweight='bold')
for i in range(1, len(data) + 1):
    c = '#eaf0fb' if i % 2 == 0 else 'white'
    for j in range(len(cols)):
        tbl[i, j].set_facecolor(c)

ax.set_title(f'Results Summary — {EXPERIMENT}',
             fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
path = FIGS_DIR / 'summary_table.pdf'
plt.savefig(str(path), bbox_inches='tight')
print(f'Saved: {path}')
plt.show()

In [ ]:
# ── Final summary ─────────────────────────────────────────────────────────
print('='*55)
print(f'  EVALUATION COMPLETE  |  {EXPERIMENT}')
print('='*55)
print(f'\nFigures saved to: outputs/{EXPERIMENT}/figures/')
for f in sorted(FIGS_DIR.glob('*.pdf')):
    print(f'  {f.name}')
print(f'\nMetrics CSV: outputs/{EXPERIMENT}/stats/final_metrics.csv')
print(f'\nShare final_metrics.csv for paper writing.')
print()
print(df[['model','accuracy','f1_macro','latency_ms',
          'size_kb','n_params']].to_string(index=False))